## **I. Présentation et exploration des données**
## I.A. Récupération des données via l'API de la NASA

In [ ]:
"""First method of data retrieval. Unfortunately, the website doesn't give the necessary permission"""
import os
import requests
import earthaccess
from zipfile import ZipFile

Entrez votre identifiant/mot de passe de EarthAccess après l'exécution de l aprochaine cellule:

In [ ]:
# 1. Login
earthaccess.login()

Une fois le prochain script exécuté, la commande 
```
tree raw_data -d
```
dans bash doit avoir la réponse suivante:
```
raw_data/
├── 2022-epi-indicators-time-series
├── 2022-epi-indicators-time-series-na
├── 2022-epi-raw-data-time-series
└── 2022-epi-raw-data-time-series-na

5 directories
```

In [ ]:
# 2. Search
results = earthaccess.search_data(
    short_name='CIESIN_SEDAC_EPI_2022',
)

# 3. Access
if results != []:
    cwd = os.getcwd()
    files = earthaccess.download(results, f"{cwd}/raw_data/")
    
    # zip folders:
    # 2022-epi-indicators-time-series-csv.zip
    # 2022-epi-indicators-time-series-na-csv.zip
    # 2022-epi-raw-data-time-series-csv.zip
    # 2022-epi-raw-data-time-series-na-csv.zip
    # 2022-epi-xlsx.zip
    
    # WARNING: in raw_data/ we also have PDF files:
    # 2022-epi-report.pdf
    # 2022-epi-technical-appendix.pdf
    # 2022-epi-policymakers-summary.pdf
    
    # Unzip the zipped folders
    zfiles = ["2022-epi-indicators-time-series-csv.zip",
              "2022-epi-indicators-time-series-na-csv.zip",
              "2022-epi-raw-data-time-series-csv.zip",
              "2022-epi-raw-data-time-series-na-csv.zip",
              "2022-epi-xlsx.zip",
            ]
    for zf in zfiles:
        full_path = f"{cwd}/raw_data/{zf}"
        relative_path = f"raw_data/{zf}"
        with ZipFile(full_path, 'r') as zObject:
            zObject.extractall(path=f"{cwd}/raw_data/")
        zObject.close()
else:
    print("\nresults is empty\n")

## I.B. Présentation des bases de données
Commençons par la liste des indices qui rentrent dans le calcul de l'EPI:

![](extra_images/fig1-3.png)

À chaque indicateur que vous voyez ici, est associé un poids comme suit:

![](extra_images/fig_ES-2.png)

Ce qu'il faut retenir sur l'ensemble des données CSV qui ont été téléchargées dans le dossier raw_data/, c'est qu'ils contiennent l'évolution temporelle pour chaque pays de chaque variable. On ne recalcule pas ces dernières, mais on en synthétise un nouvel indicateur qui va reflèter d'une réalité plus concrète que l'indicateur proposé par la NASA.

Notre indicateur se concentre sur les efforts que chaque pays fait sur le plan environnemental pour rendre son territoire plus habitable pour les humains.

Voici donc notre proposition de modification des poids qui se trouvent dans le fichier excel /home/deck/Documents/pydata_ENSAE/raw_data/2022-epi.xlsx

![](extra_images/new_inxs.png)

### Méthode de calcul:

On a supprimé toutes les sous-catégories (sc) de variables qui n'avaent pas de lien avec notre but. Il nous en restait ainsi 8, auxquelles on a ensuite assigné un entier entre 1 et 8 représentant sa priorité dans la liste des indices.

Les poids des trois catégories (c) ont été ramenés à 1/6, et on a distribué $\epsilon = 0.5$ sur les sous-catégories comme suit:
$$
poids(sc) = w \cdot W + \epsilon \cdot \delta_{0} \cdot 1.1^{8 - priorité}
$$
où
+ w: poids de la sc au sein de la c.
+ W poids de la c, soit 1/6
+ $\epsilon$: poids à départager
+ $\delta_{0}$: solution de l'équation
$$
\delta_{0} \cdot \sum_{i = 0}^{7} 1.1^{i} = 1
$$
On a donc distribué $\epsilon$ de telle façon que chaque sc pèse 10% de plus que la précédente dans la liste des priorités.

